In [ ]:
# MediaPipe 실습

# https://mediapipe-studio.webapps.google.com/demo/image_embedder

In [1]:
!pip install openai # openai 라이브러리를 설치합니다.
!pip install langchain # 랭체인 라이브러리를 설치합니다.
!pip install langchain_openai # 랭체인 라이브러리를 설치합니다.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.0 MB/s eta 0:00:00


In [2]:
import os
import openai

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('kosa')

openai.api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
#from langchain.chat_models import ChatOpenAI

In [3]:
!pip install chromadb # 벡터스토어
!pip install tiktoken # 토큰 계산용

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelemet

In [4]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/hwchase17/chat-your-data/master/state_of_the_union.txt",
    filename="state_of_the_union.txt"
)

('state_of_the_union.txt', <http.client.HTTPMessage at 0x7b1d3eeb4860>)

In [5]:
!pip install langchain

In [6]:
!pip install -U langchain-community
# -U : update
# 이미 설치된 패키지가 있을 경우 최신 버전 업데이트 하라는 의미

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-pro

In [7]:
!pip install -U langchain-chroma chromadb

In [8]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Load the document, split it into chunks, embed each chunk and load it into the vector store.

# 텍스트 문서를 로딩합니다.
raw_documents = TextLoader('state_of_the_union.txt').load()

# 청크 = 상자, 1000개 문자를 담을 수 있는 상자
# 텍스트 문서를 청크 사이즈로 자르기 위한 스플리터를 생성합니다.
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

# 청크 사이즈로 자릅니다. 텍스트 문서 > 여러개의 청크 (Document 스키마 키 page_content, metadata)
documents = text_splitter.split_documents(raw_documents)
# documents : 여러 개의 상자
print(documents)
print(documents[0].page_content)


/tmp/ipykernel_846/3353385794.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': 'state_of_the_union.txt'}, page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determin

In [9]:
# 벡터스토어 구축 (청크(문서) > 벡터화, 임베딩)
db = Chroma.from_documents(documents, OpenAIEmbeddings())

# 하나의 상자 > 1536 벡터
# 42의 상자 > 1536 * 42 = 64512

In [10]:
1536 * 42

64512

In [11]:
raw_documents

[Document(metadata={'source': 'state_of_the_union.txt'}, page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determin

In [12]:
# 총 몇 개의 상자일까?

print(len(documents))

42


In [13]:
# 첫번째 상자
documents[0]

Document(metadata={'source': 'state_of_the_union.txt'}, page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determina

In [14]:
documents[0:3]

[Document(metadata={'source': 'state_of_the_union.txt'}, page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determin

In [15]:
query = "What did the president say about Ketanji Brown Jackson"
docs = db.similarity_search(query) # k : 4 - 상위 몇 개를 가지고 올 것인가?

# 질문 > LLM > 답변
# 질문 > 임베딩 + 벡터스토어 > 관련된 문서를 반환한다.

print(docs)
print(docs[0].page_content)

[Document(id='7012cc54-2346-41b9-b649-b922ac23a2a3', metadata={'source': 'state_of_the_union.txt'}, page_content='Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. \n\nTonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. \n\nOne of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.'), Document(id='9361ec62-f3d8-4b5d-af04-55a51494ca80', metadata={'source': 'state_of_the_uni

In [16]:
docs

[Document(id='7012cc54-2346-41b9-b649-b922ac23a2a3', metadata={'source': 'state_of_the_union.txt'}, page_content='Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. \n\nTonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. \n\nOne of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.'),
 Document(id='9361ec62-f3d8-4b5d-af04-55a51494ca80', metadata={'source': 'state_of_the_un

In [17]:
# 문자열(query) > 임베딩 > 벡터 (실수의 리스트)
embedding_vector = OpenAIEmbeddings().embed_query(query)

# 벡터를 가지고 유사도 검색
docs = db.similarity_search_by_vector(embedding_vector)

print(docs[0].page_content)

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. 

Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. 

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.


In [18]:
docs

[Document(id='7012cc54-2346-41b9-b649-b922ac23a2a3', metadata={'source': 'state_of_the_union.txt'}, page_content='Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. \n\nTonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. \n\nOne of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.'),
 Document(id='9361ec62-f3d8-4b5d-af04-55a51494ca80', metadata={'source': 'state_of_the_un

In [19]:
len(embedding_vector)

1536

In [ ]:
# embedding_vector

In [ ]:
'''
문서 > 1000개의 문자를 담을 수 있는 상자에 포장 > 42개의 상자가 나옴
> 그 상자 위에 벡터(1536개 수치로 표시)

질문 > 1536개의 수치로 변환(QV) QV를 기존 42개의 상자의 벡터와 유사도 계산을 해서,
가장 유사한 상자 4개를 가지고 옴
'''

'\n문서 > 1000개의 문자를 담을 수 있는 상자에 포장 > 42개의 상자가 나옴\n> 그 상자 위에 벡터(1536개 수치로 표시)\n\n질문 > 1536개의 수치로 변환(QV) QV를 기존 42개의 상자의 벡터와 유사도 계산을 해서,\n가장 유사한 상자 4개를 가지고 옴\n'

In [20]:
!pip install openai # openai 라이브러리를 설치합니다.
!pip install langchain # 랭체인 라이브러리를 설치합니다.

In [21]:
# RAG
# LLM이 답변을 생성하기 위해, 검색(R)하고 그 검색결과를 프롬프트에 삽입(A)한 후 생성(G)

# retriever 검색기

retriever = db.as_retriever(search_kwargs={"k": 3})

# db를 쿼리를 주면, 유사한 document를 3개 주는 검색기로 사용하겠다.
# 체인에서는 retriever만 바라본다.

In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [23]:
chat_model = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
다음 문서를 참고해서 질문에 답변하세요.

문서:
{context}

질문:
{question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {
        "context": lambda question: format_docs(
            retriever.invoke(question)
        ),
        "question": lambda question: question,
    }
    | prompt
    | chat_model
    | StrOutputParser()
)

In [24]:
!pip install -U langchain-huggingface sentence-transformers

In [25]:
query = """
What did the president say about Ketanji Brown Jackson?
Write the answer in Korean.
"""

result = chain.invoke(query)

print(result)

대통령은 Ketanji Brown Jackson을 미국 대법원에 지명했다고 말했습니다. 그녀는 국가의 최고 법률 전문가 중 한 명으로, Stephen Breyer 대법관의 우수한 유산을 이어갈 것이라고 언급했습니다.


 "Write in Korean" 처럼 지시문 등의 프롬프트를 많이 작성하면, 기존 청크와의 유사성 판단에 안 좋은 영향을 미치지 않을까요?

 > 안 좋은 영향을 미칠 수 있습니다.

 * system prompt : role, few-shot, instruction
 * user prompt : query

 * db.retriever(query)